In [ ]:
from pathlib import Path

# Find the repository root so these notebooks work on other computers.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "Dataset Specs").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Dataset Specs").is_dir():
    raise FileNotFoundError("Start Jupyter from inside the Hospital Dataset project folder.")

import pandas as pd

file_path = str(PROJECT_ROOT / "Dataset Specs" / "hospital_occupancy_daily.csv")

df = pd.read_csv(file_path)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

daily_occupancy = df["occupied_beds"].asfreq("D")

print(daily_occupancy.head())
print(daily_occupancy.tail())
print(daily_occupancy.index.freq)

In [ ]:
import numpy as np

def make_lag_features(series, n_lags=7):
    df_ml = pd.DataFrame({"occupied_beds": series})
    
    #For Lag 1,2,...n
    for lag in range(1, n_lags + 1):
        df_ml[f"lag_{lag}"] = series.shift(lag)
    
    # Calendar features
    df_ml["day_of_week"] = df_ml.index.dayofweek  # 0=Mon ... 6=Sun
    df_ml["month"] = df_ml.index.month
    
    # Droprows that became NaN becuase of shifting
    df_ml = df_ml.dropna()
    return df_ml

ml_df = make_lag_features(daily_occupancy, n_lags=7)

print(ml_df.head())
print(ml_df.tail())

In [ ]:
X = ml_df.drop(columns=["occupied_beds"])
y = ml_df["occupied_beds"]

print(X.shape)
print(y.shape)

In [ ]:
data_ml = ml_df.dropna().sort_index()

train_size = int(len(data_ml) * 0.8)
train_ml = data_ml.iloc[:train_size]
test_ml  = data_ml.iloc[train_size:]

# Features (X) and Target (y)
feature_cols = ['lag_1', 'lag_7', 'day_of_week', 'month']

X_train = train_ml[feature_cols]
y_train = train_ml['occupied_beds']

X_test  = test_ml[feature_cols]
y_test  = test_ml['occupied_beds']

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train period:", X_train.index.min(), "to", X_train.index.max())
print("Test period :", X_test.index.min(), "to", X_test.index.max())

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    accuracy = 100 - mape
    return rmse, mae, mape, accuracy

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Gradient Boosting Model 1
gb_model_1 = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=3,
    random_state=42
)

gb_model_1.fit(X_train, y_train)
y_pred_gb_1 = gb_model_1.predict(X_test)

rmse1, mae1, mape1, acc1 = evaluate_model(y_test, y_pred_gb_1)

print("Gradient Boosting model-1 trained.")

In [ ]:
# Gradient Boosting Model 2 (Increasing estimators)
gb_model_2 = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=3,
    random_state=42
)

gb_model_2.fit(X_train, y_train)
y_pred_gb_2 = gb_model_2.predict(X_test)

rmse2, mae2, mape2, acc2 = evaluate_model(y_test, y_pred_gb_2)

print("Gradient Boosting model-2 trained.")

In [ ]:
# Gradient Boosting Model 3 (Increase max_depth)
gb_model_3 = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

gb_model_3.fit(X_train, y_train)
y_pred_gb_3 = gb_model_3.predict(X_test)

rmse3, mae3, mape3, acc3 = evaluate_model(y_test, y_pred_gb_3)

print("Gradient Boosting model-3 trained.")

In [ ]:
# Gradient Boosting Model 4 (Change learning rate)

gb_model_4 = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

gb_model_4.fit(X_train, y_train)
y_pred_gb_4 = gb_model_4.predict(X_test)

rmse4, mae4, mape4, acc4 = evaluate_model(y_test, y_pred_gb_4)

print("Gradient Boosting model-4 trained.")

In [ ]:
# Gradient Boosting Model 5 (Varying all parameters)
gb_model_5 = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.01,
    random_state=42
)

gb_model_5.fit(X_train, y_train)
y_pred_gb_5 = gb_model_5.predict(X_test)

rmse5, mae5, mape5, acc5 = evaluate_model(y_test, y_pred_gb_5)

print("Gradient Boosting model-5 trained.")

In [ ]:
gb_results = pd.DataFrame({
    "Model": ["GB-1", "GB-2", "GB-3", "GB-4", "GB-5"],
    "RMSE": [rmse1, rmse2, rmse3, rmse4, rmse5],
    "MAE": [mae1, mae2, mae3, mae4, mae5],
    "MAPE": [mape1, mape2, mape3, mape4, mape5],
    "Accuracy": [acc1, acc2, acc3, acc4, acc5]
})

gb_results

In [ ]:
# Plotting

import matplotlib.pyplot as plt

plt.figure(figsize=(14,8))
plt.plot(y_test.index, y_test, label="Actual", color="black", alpha=0.7)

# Plot each model prediction
#plt.plot(y_test.index, y_pred_gb_1, label="GB Model 1", alpha=0.7)
#plt.plot(y_test.index, y_pred_gb_2, label="GB Model 2", alpha=0.7)
#plt.plot(y_test.index, y_pred_gb_3, label="GB Model 3", alpha=0.7)
plt.plot(y_test.index, y_pred_gb_4, label="GB Model 4", alpha=0.7)
#plt.plot(y_test.index, y_pred_gb_5, label="GB Model 5", alpha=0.7)

plt.title("Gradient Boosting Models vs Actual")
plt.xlabel("Date")
plt.ylabel("Admissions")
plt.legend()
plt.show()

In [ ]:
from pathlib import Path

output_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/70_30"
)
output_dir.mkdir(parents=True, exist_ok=True)

gb_pred_df = pd.DataFrame({
    "date": y_test.index,
    "actual_occupied_beds": y_test.to_numpy(),
    "gb_1": y_pred_gb_1,
    "gb_2": y_pred_gb_2,
    "gb_3": y_pred_gb_3,
    "gb_4": y_pred_gb_4,
    "gb_5": y_pred_gb_5
})

gb_pred_df.to_csv(
    output_dir / "gradient_boosting_occupancy_predictions.csv",
    index=False
)

print(f"Predictions saved to: {output_dir}")